## START

In [1]:
# Load the data exactly as in homework 2:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

# This gives 72 pages.

In [2]:
len(documents) # 72

72

In [3]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [4]:
# Generating ground truth
# To evaluate search, we need a dataset of questions where we know which document is the correct answer. This is the ground truth.

# We generate it the same way as in the module. For each lesson page, we ask an LLM to write 5 questions that are answered by that page. 
# Each question is then labeled with the page it came from.

# We use the same structured-output approach as in the module - the same Questions model and the llm_structured helper from evaluation_utils.py.

# The module's instructions generate questions from a FAQ record, so we adapt them for a lesson page:

data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


## Q1 -- 2

In [5]:
# We ask for different wording from the page on purpose. Real users don't phrase their questions the way the lesson does, 
# and copying the text would make the evaluation too easy.

# For each page, build a JSON user prompt from its filename and content, then call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call also returns the token usage, the same as in the lessons.

# Q1. Generating questions
# Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

# 01-agentic-rag/lessons/01-intro.md
# 01-agentic-rag/lessons/02-environment.md
# 01-agentic-rag/lessons/03-rag.md

# Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

# What's the average number of input tokens across these 3 calls?

# 140
# 1400 == average input tokens =  1020
# 14000
# 140000

# These numbers vary between runs, even with the same model, so pick the closest option. 
# A different provider or model may land further apart, but the input tokens stay in the same 
# order of magnitude - the prompt we send is the same.

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

doc = documents[0]
# print(doc["id"])
# print(doc["content"])
# print(doc["filename"])

In [6]:

# Call the LLM for one document:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Prepare the document as JSON:

import json

user_prompt = json.dumps(doc)
# Create the messages:

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

# The parsed object is available in response.output_parsed:

result = response.output_parsed

print(result)
# We can access the list directly:

print(result.questions)
# You should see 5 questions that relate to the first FAQ document.

questions=['What is RAG trying to fix, and why do people use it with LLMs so often?', 'Why does this course build the system in plain Python instead of using a framework right away?', 'What are the main limits of an LLM that this lesson points out?', 'What will the FAQ agent in this module actually do for course students?', 'What topics are covered in the first part of the module, and how is part two different?']
['What is RAG trying to fix, and why do people use it with LLMs so often?', 'Why does this course build the system in plain Python instead of using a framework right away?', 'What are the main limits of an LLM that this lesson points out?', 'What will the FAQ agent in this module actually do for course students?', 'What topics are covered in the first part of the module, and how is part two different?']


In [7]:
for q in result.questions:
    print(q)

What is RAG trying to fix, and why do people use it with LLMs so often?
Why does this course build the system in plain Python instead of using a framework right away?
What are the main limits of an LLM that this lesson points out?
What will the FAQ agent in this module actually do for course students?
What topics are covered in the first part of the module, and how is part two different?


In [8]:
# llm_structured: calls the OpenAI API with structured output
# llm_structured_retry: retries structured-output calls when a request fails
# calc_price: calculates the price from token usage
# calc_total_price: calculates the total price from multiple usage objects
# map_progress: runs work in parallel and tracks progress. We'll use it in the next lesson.
# Import the structured-output helper:

from evaluation_utils import llm_structured
# Use it on the same document:

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

# Import the price helper:

from evaluation_utils import calc_price
# Calculate the cost of this call:

cost = calc_price(usage)

cost, usage, usage.input_tokens
print('usage.input_tokens = ' , usage.input_tokens)

['What problem does retrieval-augmented generation solve compared with using an LLM by itself?', 'Why does the course build the RAG system in plain Python instead of using a framework right away?', 'What are the main limits of an LLM that make RAG useful?', 'What is the FAQ agent in this module supposed to do for students of the course?', 'What will be covered in the first part of the module, and how is the second part different?']
usage.input_tokens =  1020


In [9]:
q1_docs = documents[:3]
its = 0 # sum of input tokens

for q in q1_docs:
    print(q['filename'])
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    print(result.questions)
    print('usage.input_tokens = ' , usage.input_tokens)
    its += usage.input_tokens

average = its/3
print('average input tokens = ', average)

01-agentic-rag/lessons/01-intro.md
['What is Retrieval-Augmented Generation, and why does it help with answers the model may not know on its own?', 'Why does this course build the RAG system in plain Python instead of using a framework right away?', 'What are the main weaknesses of LLMs that this module is trying to work around?', 'What will you build in this module to make the RAG idea concrete?', 'How are the two parts of the module different from each other?']
usage.input_tokens =  1020
01-agentic-rag/lessons/02-environment.md
["What exactly is a RAG system, and how does it help when the model doesn't know the answer on its own?", 'Why are LLMs described as black boxes in this course instead of studying how they work internally?', 'What are the main limits of LLMs that make retrieval-based setup necessary?', 'What is the course building as the example RAG project, and what kind of questions should it answer?', 'What topics are covered in the first part of the module, and what change

## Q2 -- 2

In [13]:
# The full ground truth
# You don't need to generate the data for the rest of the homework. We already did it for all 72 pages, 
# using the same approach as in the lessons, and saved the 360 questions to a file.

# Download it:

# PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
# Load it with pandas into a list of records called ground_truth. Each record has a question and the filename of the page that should answer it.

import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth) # 360

360

In [14]:
# Searching the chunks
# We search over the same chunks as in homework 2.

# Create them with chunk_documents:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks) # This gives 295 chunks.

295

In [15]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [17]:
query = "How do I store vectors in PostgreSQL?"

In [21]:
# Now rebuild the search from homework 2 over these chunks. Build a text index (Index) and a vector index (VectorSearch), 
# both keyed on filename. Wrap each one in a function, text_search and vector_search, that takes a query and the number 
# of results to return (5 by default).


# Build a text index (Index)
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(
        query,
        # boost_dict={"question": 2.0, "section": 0.5},
        # filter_dict={"course": "llm-zoomcamp"},
        num_results=num_results
    )

search_results = text_search(query)
for r in search_results:
    print(r['filename'])


02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [33]:
# Build a vector index (VectorSearch), keyed on filename. Wrap in a function, vector_search, 
# that takes a query and the number of results to return (5 by default).
from tqdm.auto import tqdm
from embedder import Embedder

embed = Embedder()

# STEP 2 - embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:
vectors = []

for i in tqdm(range(len(chunks))):
    chunk_vector = embed.encode(chunks[i]['content'])
    vectors.append(chunk_vector)

print(len(chunks))
print(len(vectors))

import numpy as np
X = np.array(vectors)
print(X.shape)

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)


  0%|          | 0/295 [00:00<?, ?it/s]

295
295
(295, 384)


In [35]:
def vector_search(query, num_results=5):
    query_vector = embed.encode(query)
    results = vindex.search(query_vector, num_results=5)
    return results


results = vector_search(query)
# results

In [36]:
# For hybrid search, reuse the rrf function from homework 2:

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [37]:
# Then define hybrid_search on top of it:

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [38]:
# Q2. First result with text search
# Take the first question from the ground truth:

q = ground_truth[0]["question"]
# After running text_search for it, what's the filename of the first result?

# 01-agentic-rag/lessons/01-intro.md
# 01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/03-rag.md
# 01-agentic-rag/lessons/13-function-calling.md
# 01-agentic-rag/lessons/10-rag-next-steps.md
print(q)

search_results = text_search(q)
for r in search_results:
    print(r['filename'])

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md


## Q3 -- 1

In [43]:
# Q3. First result with vector search
# After running vector_search for the same question, what's the filename of the first result?

# 01-agentic-rag/lessons/01-intro.md == 01-agentic-rag/lessons/01-intro.md

# 01-agentic-rag/lessons/03-rag.md
# 04-evaluation/lessons/11-evaluation-intro.md
# 04-evaluation/lessons/12-rag-answers.md

# This question was generated from 01-agentic-rag/lessons/01-intro.md. Notice that one method finds the right page at the top and the other doesn't. 
# That's exactly why we measure across the whole dataset instead of trusting one query.
q = ground_truth[0]["question"]
print(q)

search_results = vector_search(q)
for r in search_results:
    print(r['filename'])


What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/11-evaluation-intro.md
04-evaluation/lessons/12-rag-answers.md
01-agentic-rag/lessons/10-rag-next-steps.md
06-best-practices/lessons/01-intro.md


## Q4 --

In [ ]:
# Evaluation metrics
# We evaluate search exactly as in the module, reusing the same functions from the lecture. We change only the label. 
# Our ground truth uses filename, so a result counts as a hit when a returned chunk's filename matches the question's filename, not a document id.

# As a reminder, these functions do the following:

# compute_relevance runs search for a question and returns a list of 0s and 1s
# hit_rate is the fraction of questions where the correct page appears in the results
# mrr (Mean Reciprocal Rank) also rewards finding the page near the top
# evaluate runs a search function over the whole ground truth and returns both metrics

